<a href="https://colab.research.google.com/github/uncertain-uncertainity/Learning/blob/main/03_feature_engineering/scikit_learn_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Scikit-learn Pipelines

A Pipeline combines multiple preprocessing steps and a final estimator into one object.

Advantages:
- Makes code cleaner and reusable.
- Ensures preprocessing is applied consistently.
- Helps prevent data leakage when fitted correctly.
- Works with cross-validation and hyperparameter tuning.

Syntax:
Pipeline([
    ("step_name_1", transformer_1),
    ("step_name_2", transformer_2),
    ("model", estimator)
])


In [9]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Create sample dataset
df = pd.DataFrame({
    "Age": [22, 25, np.nan, 30, 28, 35, 40, 26, 32, 24, 29, 38],
    "Fare": [100, 200, 150, np.nan, 300, 250, 180, 120, 220, 90, 160, 280],
    "Survived": [0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0]
})

X = df[["Age", "Fare"]]
y = df["Survived"]

# Split data before fitting preprocessing steps
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Pipeline: imputation -> scaling -> model
# Fit on training data only to avoid data leakage.
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Accuracy:", pipe.score(X_test, y_test))


Accuracy: 0.3333333333333333


In [15]:
# Cell 2 — Pipeline with Feature Selection
from sklearn.feature_selection import SelectKBest, f_classif

# SelectKBest selects the top k features.
# f_classif uses ANOVA F-scores for classification.
# k=1 selects one feature; choose k using cross-validation.

pipe_fs = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selection", SelectKBest(score_func=f_classif, k=1)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_fs.fit(X_train, y_train)
y_pred_fs = pipe_fs.predict(X_test)

print("Accuracy:", pipe_fs.score(X_test, y_test))

# Display selected feature names (X_train must be a DataFrame).
selected = X_train.columns[
    pipe_fs.named_steps["selection"].get_support()
]
print("Selected features:", selected.tolist())


Accuracy: 0.3333333333333333
Selected features: ['Age']


In [11]:
# Cell 3 — Cross-validation
from sklearn.model_selection import cross_val_score

# Cross-validation gives a more reliable estimate than one split.
# Use training data for model selection, not the test set.

scores = cross_val_score(
    pipe_fs, X_train, y_train, cv=3, scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean CV accuracy:", scores.mean())


Fold scores: [0.33333333 1.         0.33333333]
Mean CV accuracy: 0.5555555555555555


In [16]:
# Cell 4 — Visualize Pipeline
from sklearn import set_config

set_config(display="diagram")
pipe_fs


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('selection', SelectKBest(k=1)), ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])